# [11.1] PCA, SVD, and Geometry Controls

Build local representation-geometry checks before trusting a visualization: centered PCA/SVD, held-out label prediction, white-noise controls, stability controls, causal-direction checks, and template centering.

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter11_representation_geometry"
section = "part1_pca_svd_geometry_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_pca_svd_geometry_controls.tests as tests
import part1_pca_svd_geometry_controls.utils as utils

GT_TIER = "GT-0"
EXERCISE_ID = "11_1_pca_svd_and_geometry_controls"
EXPECTED_RUNTIME = "35-45 minutes for exercises; about 1 minute for the CUDA preflight"
REQUIRES_GPU = True

CausalDirection = Literal["increase", "decrease"]


@dataclass(frozen=True)
class PCAProjection:
    projected: t.Tensor
    components: t.Tensor
    explained_variance_ratio: t.Tensor


@dataclass(frozen=True)
class GeometryLabelPredictionReport:
    heldout_accuracy: float
    predicts_heldout_labels: bool


@dataclass(frozen=True)
class WhiteNoiseControlReport:
    real_accuracy: float
    noise_accuracy: float
    margin: float
    survives_white_noise_control: bool


@dataclass(frozen=True)
class GeometryStabilityReport:
    neighbor_sets: tuple[tuple[int, ...], ...]
    mean_pairwise_jaccard: float
    stable_across_seeds: bool


@dataclass(frozen=True)
class DirectionCausalEffectReport:
    baseline_mean: float
    intervened_mean: float
    random_control_mean: float
    observed_delta: float
    random_delta: float
    has_causal_effect: bool


## Exercise 1 - PCA via SVD

Center activations, run SVD, keep right-singular vectors as components, and compute explained-variance ratios from squared singular values.

In [ ]:
def pca_svd_projection(
    activations: t.Tensor,
    *,
    n_components: int = 2,
) -> PCAProjection:
    raise NotImplementedError()


tests.test_pca_svd_projection_centers_and_reports_variance(pca_svd_projection)

## Exercise 2 - Held-out label prediction

Fit nearest centroids on training points only, then evaluate held-out labels.

In [ ]:
def nearest_centroid_predictions(
    train_points: t.Tensor,
    train_labels: t.Tensor,
    heldout_points: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


def geometry_label_prediction_report(
    train_points: t.Tensor,
    train_labels: t.Tensor,
    heldout_points: t.Tensor,
    heldout_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> GeometryLabelPredictionReport:
    raise NotImplementedError()


tests.test_geometry_label_prediction_report_uses_heldout_centroids(
    geometry_label_prediction_report,
)

## Exercise 3 - White-noise control

Require the real geometry to beat a white-noise baseline by a declared margin.

In [ ]:
def white_noise_control_report(
    *,
    real_accuracy: float,
    noise_accuracy: float,
    min_margin: float = 0.2,
) -> WhiteNoiseControlReport:
    raise NotImplementedError()


tests.test_white_noise_control_report_requires_margin(white_noise_control_report)

## Exercise 4 - Stability control

Average all pairwise Jaccard overlaps between seed-level neighbor sets.

In [ ]:
def geometry_stability_report(
    neighbor_sets: list[list[int]],
    *,
    min_jaccard: float = 0.5,
) -> GeometryStabilityReport:
    raise NotImplementedError()


tests.test_geometry_stability_report_averages_pairwise_jaccard(
    geometry_stability_report,
)

## Exercise 5 - Causal direction control

A direction must move the target in the expected direction and beat a random-direction control.

In [ ]:
def direction_causal_effect_report(
    baseline_scores: t.Tensor,
    intervened_scores: t.Tensor,
    random_control_scores: t.Tensor,
    *,
    expected_direction: CausalDirection = "increase",
    min_effect: float = 0.2,
    min_random_margin: float = 0.1,
) -> DirectionCausalEffectReport:
    raise NotImplementedError()


tests.test_direction_causal_effect_report_beats_random_control(
    direction_causal_effect_report,
)

## Exercise 6 - Template centering

Remove each prompt-template mean direction across examples.

In [ ]:
def template_center_activations(
    activations_by_template: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_template_center_activations_removes_each_template_mean(
    template_center_activations,
)

## Exercise 7 - Notebook contract

Assemble the visible smoke-test contract from your implementations.

In [ ]:
def pca_smoke_test() -> dict:
    activations = t.tensor(
        [
            [2.0, 0.0],
            [1.0, 0.0],
            [-1.0, 0.0],
            [-2.0, 0.0],
        ]
    )
    projection = pca_svd_projection(activations, n_components=1)
    return {
        "projected_shape": list(projection.projected.shape),
        "components_shape": list(projection.components.shape),
        "explained_variance_ratio": [
            round(value, 6)
            for value in projection.explained_variance_ratio.tolist()
        ],
    }


def prediction_smoke_test() -> dict:
    train_points = t.tensor([[0.0, 0.0], [0.2, 0.0], [2.0, 0.0], [2.2, 0.0]])
    train_labels = t.tensor([0, 0, 1, 1])
    heldout_points = t.tensor([[0.1, 0.0], [2.1, 0.0]])
    heldout_labels = t.tensor([0, 1])
    return geometry_label_prediction_report(
        train_points,
        train_labels,
        heldout_points,
        heldout_labels,
        min_accuracy=1.0,
    ).__dict__


def noise_control_smoke_test() -> dict:
    return white_noise_control_report(
        real_accuracy=0.9,
        noise_accuracy=0.5,
        min_margin=0.2,
    ).__dict__


def stability_smoke_test() -> dict:
    return geometry_stability_report(
        [[1, 2, 3], [1, 2, 4], [1, 2, 3]],
        min_jaccard=0.5,
    ).__dict__


def causal_direction_smoke_test() -> dict:
    baseline = t.tensor([0.2, 0.3])
    intervened = t.tensor([0.8, 0.7])
    random_control = t.tensor([0.35, 0.25])
    return direction_causal_effect_report(
        baseline,
        intervened,
        random_control,
        expected_direction="increase",
        min_effect=0.4,
        min_random_margin=0.2,
    ).__dict__


def template_centering_smoke_test() -> dict:
    activations = t.tensor(
        [
            [[10.0, 1.0], [12.0, 3.0]],
            [[-5.0, 2.0], [-1.0, 4.0]],
        ]
    )
    centered = template_center_activations(activations)
    return {
        "shape": list(centered.shape),
        "max_template_mean_abs": centered.mean(dim=1).abs().max().item(),
        "first_template_centered": centered[0].tolist(),
    }


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "pca": pca_smoke_test(),
        "prediction": prediction_smoke_test(),
        "noise_control": noise_control_smoke_test(),
        "stability": stability_smoke_test(),
        "causal_direction": causal_direction_smoke_test(),
        "template_centering": template_centering_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    tests.test_committed_verification_report_has_visualization_sweep_controls()
    gpu = report["metrics"]["gpu_test"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "pythia_weekday_visualization_passed",
    "pythia_month_visualization_passed",
    "pythia_visualization_seed_count",
    "pythia_visualization_setting_count",
    "pythia_weekday_umap_min_heldout_knn_accuracy",
    "pythia_month_umap_min_heldout_knn_accuracy",
    "pythia_weekday_umap_min_trustworthiness",
    "pythia_month_umap_min_trustworthiness",
    "peak_vram_gb",
]}
